In [ ]:
import pandas as pd
import numpy as np


In [ ]:
df = pd.read_parquet('bureau_balance.parquet')
print('Shape:', df.shape)
print('Уникальных кредитов (SK_ID_BUREAU):', df['SK_ID_BUREAU'].nunique())
print('\nПропуски топ-20:')
missing = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
print(missing[missing > 0].head(20) if missing[missing > 0].any() else 'Пропусков нет')
print('\nЧисло категориальных фич:')
print(df.select_dtypes(include='object').shape[1])
print('\nЧисло числовых фич:')
print(df.select_dtypes(include='number').shape[1])

## Описание таблицы

- 1 строка = 1 месяц истории по 1 кредиту из ЦКИ
- `SK_ID_BUREAU` — ID кредита (связь с `bureau.parquet`)
- `MONTHS_BALANCE` — месяц относительно даты заявки (0 = месяц заявки, -1 = предыдущий и т.д.)
- `STATUS` — статус кредита в этот месяц:
  - `C` = кредит закрыт
  - `X` = статус неизвестен
  - `0` = нет просрочки
  - `1`-`5` = просрочка 1–5 DPD-категорий (чем больше — тем глубже)

In [ ]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f'Категориальных фич: {len(cat_cols)}')
for col in cat_cols:
    print(f'\n  {col}: {df[col].nunique()} уникальных')
    print(df[col].value_counts().to_string())

print('\nMONTHS_BALANCE:')
print(df['MONTHS_BALANCE'].describe())

**Выводы:**
- `STATUS` — 8 уникальных значений. C+X+0 = «нет проблем» (~97.4%), 1–5 = просрочка (~2.6%)
- `MONTHS_BALANCE` — от -96 до 0, т.е. история до 8 лет назад
- Пропусков нет — таблица полная

## Очистка и кодирование STATUS

In [ ]:
# Переводим STATUS в числовую глубину просрочки:
# C / X / 0 = нет просрочки (0), 1-5 = глубина DPD
status_map = {'C': 0, 'X': 0, '0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5}
df['STATUS_NUM'] = df['STATUS'].map(status_map)

print('Проверка STATUS_NUM:')
print(df.groupby('STATUS')['STATUS_NUM'].first())

print('\nДоля месяцев с просрочкой (STATUS_NUM > 0):')
overdue_rate = (df['STATUS_NUM'] > 0).mean()
print(f'  {overdue_rate*100:.2f}%')

## Агрегация на уровень SK_ID_BUREAU

In [ ]:
bb_agg = df.groupby('SK_ID_BUREAU', sort=False).agg(
    bb_count         = ('MONTHS_BALANCE', 'count'),                          # длина истории (мес)
    bb_months_min    = ('MONTHS_BALANCE', 'min'),                            # насколько давно открыт
    bb_dpd_max       = ('STATUS_NUM',     'max'),                            # макс глубина просрочки
    bb_dpd_mean      = ('STATUS_NUM',     'mean'),                           # средняя глубина
    bb_overdue_any   = ('STATUS_NUM',     lambda x: int((x > 0).any())),     # хотя бы раз просрочил
).reset_index()

print('Shape bb_agg:', bb_agg.shape)
print('\nПропуски в bb_agg (%):')
missing_agg = (bb_agg.isnull().sum() / len(bb_agg) * 100).sort_values(ascending=False)
print(missing_agg[missing_agg > 0] if missing_agg[missing_agg > 0].any() else 'Пропусков нет')
print('\nПервые строки:')
print(bb_agg.head())

In [ ]:
print('Число категориальных фич:', bb_agg.select_dtypes(include='object').shape[1])
print('Число числовых фич:', bb_agg.select_dtypes(include='number').shape[1] - 1)  # -1 за SK_ID_BUREAU

print('\nDescribe bb_agg:')
print(bb_agg.drop(columns='SK_ID_BUREAU').describe().T[['mean', 'std', 'min', '25%', '50%', '75%', 'max']].round(3))

In [ ]:
print('Статистика bb_dpd_max (глубина просрочки):')
print(bb_agg['bb_dpd_max'].value_counts().sort_index())

print('\nДоля кредитов с хотя бы одной просрочкой (bb_overdue_any):')
print(bb_agg['bb_overdue_any'].value_counts(normalize=True))

print('\nРаспределение длины истории (bb_count):')
print(bb_agg['bb_count'].describe())

In [ ]:
bb_agg.to_csv('bb_agg.csv', index=False)
print(f'Сохранено: bb_agg.csv  —  {bb_agg.shape[0]} строк, {bb_agg.shape[1]} колонок')
print('Используется в bureau_EDA.ipynb для мерджа через SK_ID_BUREAU')